In [1]:
import math
import random
from statistics import mean, stdev


# ============================================================
# POKER BANKROLL RISK ENGINE
# ============================================================
#
# Inputs:
#   - Session length
#   - Session profit/loss
#   - Current bankroll
#   - Hours/week
#   - Target bankroll
#
# Calculates:
#   - Observed win rate
#   - Hourly standard deviation
#   - Variance
#   - Theoretical probability of ruin
#   - Required bankroll
#   - 95% confidence interval for win rate
#   - Monte Carlo expected bankroll
#   - Probability of reaching target
#   - Maximum downswing distribution
#
# No external packages required.
# ============================================================


# ------------------------------------------------------------
# INPUT HELPERS
# ------------------------------------------------------------

def get_float(prompt, allow_negative=True):
    """Safely get a floating-point number from the user."""

    while True:
        try:
            value = float(input(prompt))

            if not allow_negative and value < 0:
                print("Please enter a number >= 0.")
                continue

            return value

        except ValueError:
            print("Please enter a valid number.")


def get_positive_float(prompt):
    """Get a positive floating-point number."""

    while True:
        value = get_float(prompt, allow_negative=False)

        if value <= 0:
            print("Please enter a number greater than 0.")
        else:
            return value


def get_positive_int(prompt):
    """Get a positive integer."""

    while True:
        try:
            value = int(input(prompt))

            if value <= 0:
                print("Please enter a whole number greater than 0.")

            else:
                return value

        except ValueError:
            print("Please enter a whole number.")


# ------------------------------------------------------------
# SESSION DATA
# ------------------------------------------------------------

def collect_sessions():
    """
    Collect session length and profit/loss.

    Returns:
        list of dictionaries containing:
            hours
            profit
            hourly_result
    """

    print("\n" + "=" * 60)
    print("SESSION HISTORY")
    print("=" * 60)

    number_of_sessions = get_positive_int(
        "How many sessions would you like to enter? "
    )

    sessions = []

    for i in range(number_of_sessions):

        print(f"\nSession {i + 1}")

        hours = get_positive_float(
            "  Session length (hours): "
        )

        profit = get_float(
            "  Profit/loss ($): "
        )

        hourly_result = profit / hours

        sessions.append({
            "hours": hours,
            "profit": profit,
            "hourly_result": hourly_result
        })

    return sessions


# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

def calculate_statistics(sessions):
    """
    Calculate poker statistics from session history.

    Standard deviation is calculated from hourly session results.
    """

    hourly_results = [
        session["hourly_result"]
        for session in sessions
    ]

    total_hours = sum(
        session["hours"]
        for session in sessions
    )

    total_profit = sum(
        session["profit"]
        for session in sessions
    )

    # Weighted hourly win rate.
    #
    # Example:
    # $500 profit / 20 hours = $25/hr
    #
    # This is preferable to simply averaging the individual
    # session hourly rates because sessions may have different
    # lengths.
    win_rate = total_profit / total_hours

    # Sample standard deviation of hourly session results.
    if len(hourly_results) >= 2:
        hourly_sd = stdev(hourly_results)
    else:
        hourly_sd = 0

    variance = hourly_sd ** 2

    return {
        "hourly_results": hourly_results,
        "total_hours": total_hours,
        "total_profit": total_profit,
        "win_rate": win_rate,
        "standard_deviation": hourly_sd,
        "variance": variance
    }


# ------------------------------------------------------------
# PROBABILITY OF RUIN
# ------------------------------------------------------------

def probability_of_ruin(bankroll, win_rate, variance):
    """
    Continuous approximation of lifetime probability of ruin.

        P(Ruin) = exp(-2 * win_rate * bankroll / variance)

    This assumes:
        - Positive expected return
        - Constant win rate
        - Constant variance
        - Fixed stakes
        - No deposits/withdrawals
    """

    if bankroll <= 0:
        return 1.0

    if win_rate <= 0:
        return 1.0

    if variance <= 0:
        return 0.0

    result = math.exp(
        -2 * win_rate * bankroll / variance
    )

    return min(max(result, 0), 1)


# ------------------------------------------------------------
# REQUIRED BANKROLL
# ------------------------------------------------------------

def required_bankroll(win_rate, standard_deviation, ruin_probability):
    """
    Calculate bankroll required to achieve a target probability
    of lifetime ruin.

        B = -(variance / (2 * win_rate)) * ln(RoR)
    """

    if win_rate <= 0:
        return float("inf")

    variance = standard_deviation ** 2

    if variance == 0:
        return 0

    bankroll = (
        -(variance / (2 * win_rate))
        * math.log(ruin_probability)
    )

    return bankroll


# ------------------------------------------------------------
# CONFIDENCE INTERVAL
# ------------------------------------------------------------

def confidence_interval(hourly_results):
    """
    Approximate 95% confidence interval for the mean hourly result.
    """

    n = len(hourly_results)

    if n < 2:
        return None

    sample_mean = mean(hourly_results)
    sample_sd = stdev(hourly_results)

    # 95% normal approximation.
    z = 1.96

    standard_error = sample_sd / math.sqrt(n)

    margin = z * standard_error

    lower = sample_mean - margin
    upper = sample_mean + margin

    return lower, upper


# ------------------------------------------------------------
# MONTE CARLO SIMULATION
# ------------------------------------------------------------

def monte_carlo(
    starting_bankroll,
    hourly_results,
    hours_per_week,
    weeks=52,
    simulations=10000,
    target_bankroll=None
):
    """
    Bootstrap Monte Carlo simulation.

    Instead of assuming a normal distribution, each simulated
    hour randomly selects an observed hourly result from the
    player's actual session history.

    This makes the simulation based on the player's observed data.
    """

    total_hours = round(
        hours_per_week * weeks
    )

    final_bankrolls = []

    maximum_drawdowns = []

    target_reached = 0

    ruined = 0

    for simulation in range(simulations):

        bankroll = starting_bankroll

        peak = starting_bankroll

        maximum_drawdown = 0

        reached_target = False

        for hour in range(total_hours):

            result = random.choice(hourly_results)

            bankroll += result

            # Ruin
            if bankroll <= 0:
                bankroll = 0
                ruined += 1
                break

            # New bankroll high
            if bankroll > peak:
                peak = bankroll

            # Current drawdown
            drawdown = peak - bankroll

            if drawdown > maximum_drawdown:
                maximum_drawdown = drawdown

            # Target reached
            if (
                target_bankroll is not None
                and bankroll >= target_bankroll
            ):
                reached_target = True

        final_bankrolls.append(bankroll)

        maximum_drawdowns.append(
            maximum_drawdown
        )

        if reached_target:
            target_reached += 1

    final_bankrolls.sort()
    maximum_drawdowns.sort()

    return {
        "final_bankrolls": final_bankrolls,
        "maximum_drawdowns": maximum_drawdowns,
        "target_probability":
            target_reached / simulations
            if target_bankroll is not None
            else None,
        "simulation_ruin":
            ruined / simulations
    }


# ------------------------------------------------------------
# PERCENTILE
# ------------------------------------------------------------

def percentile(values, percentile):
    """
    Calculate percentile using linear interpolation.
    """

    if not values:
        return None

    index = (len(values) - 1) * percentile

    lower = math.floor(index)
    upper = math.ceil(index)

    if lower == upper:
        return values[lower]

    weight = index - lower

    return (
        values[lower] * (1 - weight)
        + values[upper] * weight
    )


# ------------------------------------------------------------
# FORMATTING
# ------------------------------------------------------------

def dollars(value):
    """Format a number as dollars."""

    if math.isinf(value):
        return "N/A"

    return f"${value:,.2f}"


# ------------------------------------------------------------
# MAIN PROGRAM
# ------------------------------------------------------------

def main():

    print("\n")
    print("=" * 60)
    print("             POKER BANKROLL RISK ENGINE")
    print("=" * 60)

    print(
        "\nThis tool estimates your poker volatility from your"
        "\nactual session history."
    )

    print(
        "\nFor each session, enter:"
        "\n  1. Session length"
        "\n  2. Profit/loss"
    )

    # --------------------------------------------------------
    # SESSION DATA
    # --------------------------------------------------------

    sessions = collect_sessions()

    stats = calculate_statistics(sessions)

    hourly_results = stats["hourly_results"]
    total_hours = stats["total_hours"]
    total_profit = stats["total_profit"]
    win_rate = stats["win_rate"]
    hourly_sd = stats["standard_deviation"]
    variance = stats["variance"]

    # --------------------------------------------------------
    # BANKROLL INPUT
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("BANKROLL PARAMETERS")
    print("=" * 60)

    stake = input(
        "Stake (example: $1/$3): "
    )

    bankroll = get_positive_float(
        "Current bankroll ($): "
    )

    hours_per_week = get_positive_float(
        "Hours played per week: "
    )

    target_bankroll = get_positive_float(
        "Target bankroll ($): "
    )

    # --------------------------------------------------------
    # RISK TOLERANCE
    # --------------------------------------------------------

    print("\nDesired maximum lifetime risk of ruin:")

    print("1 = 1%")
    print("2 = 2%")
    print("5 = 5%")
    print("10 = 10%")

    ruin_percent = get_positive_float(
        "Enter desired percentage: "
    )

    desired_ruin = ruin_percent / 100

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    theoretical_ror = probability_of_ruin(
        bankroll,
        win_rate,
        variance
    )

    bankroll_1 = required_bankroll(
        win_rate,
        hourly_sd,
        0.01
    )

    bankroll_5 = required_bankroll(
        win_rate,
        hourly_sd,
        0.05
    )

    bankroll_10 = required_bankroll(
        win_rate,
        hourly_sd,
        0.10
    )

    bankroll_custom = required_bankroll(
        win_rate,
        hourly_sd,
        desired_ruin
    )

    ci = confidence_interval(
        hourly_results
    )

    # --------------------------------------------------------
    # MONTE CARLO
    # --------------------------------------------------------

    simulation = monte_carlo(
        starting_bankroll=bankroll,
        hourly_results=hourly_results,
        hours_per_week=hours_per_week,
        weeks=52,
        simulations=10000,
        target_bankroll=target_bankroll
    )

    final_bankrolls = simulation[
        "final_bankrolls"
    ]

    drawdowns = simulation[
        "maximum_drawdowns"
    ]

    # --------------------------------------------------------
    # REPORT
    # --------------------------------------------------------

    print("\n\n")
    print("=" * 60)
    print("                     RISK REPORT")
    print("=" * 60)

    print(f"\nStake:              {stake}")

    print("\n--- SESSION DATA ---")

    print(
        f"Sessions analyzed: {len(sessions)}"
    )

    print(
        f"Total hours:       {total_hours:.1f}"
    )

    print(
        f"Total profit/loss: {dollars(total_profit)}"
    )

    print("\n--- PERFORMANCE ---")

    print(
        f"Win rate:           {dollars(win_rate)}/hr"
    )

    print(
        f"Standard deviation: {dollars(hourly_sd)}/hr"
    )

    print(
        f"Variance:           {variance:,.2f}"
    )

    # --------------------------------------------------------
    # CONFIDENCE INTERVAL
    # --------------------------------------------------------

    print("\n--- WIN RATE UNCERTAINTY ---")

    if ci:

        print(
            f"Approx. 95% CI:     "
            f"{dollars(ci[0])}/hr to "
            f"{dollars(ci[1])}/hr"
        )

    else:

        print(
            "Not enough sessions to estimate a CI."
        )

    # --------------------------------------------------------
    # RISK OF RUIN
    # --------------------------------------------------------

    print("\n--- RISK OF RUIN ---")

    print(
        f"Current bankroll:   {dollars(bankroll)}"
    )

    print(
        f"Estimated lifetime RoR: "
        f"{theoretical_ror:.2%}"
    )

    # --------------------------------------------------------
    # REQUIRED BANKROLL
    # --------------------------------------------------------

    print("\n--- REQUIRED BANKROLL ---")

    print(
        f"1% lifetime RoR:    {dollars(bankroll_1)}"
    )

    print(
        f"5% lifetime RoR:    {dollars(bankroll_5)}"
    )

    print(
        f"10% lifetime RoR:   {dollars(bankroll_10)}"
    )

    print(
        f"{ruin_percent:g}% lifetime RoR:   "
        f"{dollars(bankroll_custom)}"
    )

    # --------------------------------------------------------
    # MONTE CARLO
    # --------------------------------------------------------

    print("\n--- 1-YEAR MONTE CARLO ---")

    print(
        "Simulations:        10,000"
    )

    print(
        f"Hours simulated:    "
        f"{round(hours_per_week * 52):,}"
    )

    print(
        f"10th percentile:    "
        f"{dollars(percentile(final_bankrolls, 0.10))}"
    )

    print(
        f"25th percentile:    "
        f"{dollars(percentile(final_bankrolls, 0.25))}"
    )

    print(
        f"Median bankroll:    "
        f"{dollars(percentile(final_bankrolls, 0.50))}"
    )

    print(
        f"Expected bankroll:  "
        f"{dollars(mean(final_bankrolls))}"
    )

    print(
        f"75th percentile:    "
        f"{dollars(percentile(final_bankrolls, 0.75))}"
    )

    print(
        f"90th percentile:    "
        f"{dollars(percentile(final_bankrolls, 0.90))}"
    )

    # --------------------------------------------------------
    # TARGET PROBABILITY
    # --------------------------------------------------------

    print("\n--- TARGET BANKROLL ---")

    print(
        f"Target:             "
        f"{dollars(target_bankroll)}"
    )

    print(
        f"Probability of "
        f"reaching target within 1 year: "
        f"{simulation['target_probability']:.2%}"
    )

    # --------------------------------------------------------
    # DOWNSWINGS
    # --------------------------------------------------------

    print("\n--- DOWNSWING DISTRIBUTION ---")

    print(
        f"Median maximum downswing: "
        f"{dollars(percentile(drawdowns, 0.50))}"
    )

    print(
        f"75th percentile:           "
        f"{dollars(percentile(drawdowns, 0.75))}"
    )

    print(
        f"90th percentile:           "
        f"{dollars(percentile(drawdowns, 0.90))}"
    )

    # --------------------------------------------------------
    # WARNING
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("IMPORTANT MODEL NOTES")
    print("=" * 60)

    print(
        "\n1. Standard deviation is estimated from the session"
        "\n   results entered by the user."
    )

    print(
        "\n2. A small sample can produce a very unreliable SD."
    )

    print(
        "\n3. The Monte Carlo simulation bootstraps the player's"
        "\n   observed hourly results."
    )

    print(
        "\n4. Theoretical risk of ruin assumes a constant win rate,"
        "\n   constant variance, and fixed stakes."
    )

    print(
        "\n5. These are statistical estimates, not guarantees."
    )

    print("\n" + "=" * 60)


if __name__ == "__main__":
    main()



             POKER BANKROLL RISK ENGINE

This tool estimates your poker volatility from your
actual session history.

For each session, enter:
  1. Session length
  2. Profit/loss

SESSION HISTORY
How many sessions would you like to enter? 2

Session 1
  Session length (hours): 1
  Profit/loss ($): 20

Session 2
  Session length (hours): 3
  Profit/loss ($): 100

BANKROLL PARAMETERS
Stake (example: $1/$3): 1/3
Current bankroll ($): 100
Hours played per week: 10
Target bankroll ($): 500

Desired maximum lifetime risk of ruin:
1 = 1%
2 = 2%
5 = 5%
10 = 10%
Enter desired percentage: 2



                     RISK REPORT

Stake:              1/3

--- SESSION DATA ---
Sessions analyzed: 2
Total hours:       4.0
Total profit/loss: $120.00

--- PERFORMANCE ---
Win rate:           $30.00/hr
Standard deviation: $9.43/hr
Variance:           88.89

--- WIN RATE UNCERTAINTY ---
Approx. 95% CI:     $13.60/hr to $39.73/hr

--- RISK OF RUIN ---
Current bankroll:   $100.00
Estimated lifetime RoR: 0.